In [2]:
import requests
from bs4 import BeautifulSoup

print("requests OK:", requests.__version__)
print("BeautifulSoup OK")

requests OK: 2.33.1
BeautifulSoup OK


¿Qué estamos haciendo acá?

requests.get(url) → le manda una petición HTTP GET al servidor
response.status_code → el servidor responde con un código. 200 = éxito, 404 = no encontrado, 500 = error del servidor
response.text → el HTML crudo que devolvió el servidor, como un string gigante

In [3]:
url = "http://books.toscrape.com/"

response = requests.get(url)

print("Status code:", response.status_code)
print("Tipo de respuesta:", type(response.text))
print("Primeros 200 caracteres del HTML:")
print(response.text[:200])

Status code: 200
Tipo de respuesta: <class 'str'>
Primeros 200 caracteres del HTML:
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if I


Paso 3 — Parsear el HTML con BeautifulSoup
Tenemos el HTML como string gigante, pero así no podemos buscar nada fácilmente. BeautifulSoup lo convierte en un árbol navegable.
En una celda nueva:
¿Qué es el árbol HTML?
El HTML tiene una estructura jerárquica:
html
 └── body
      └── div
           └── h1 → "Books to Scrape"
           └── ul
                └── li → "Mystery"
                └── li → "Travel"
BeautifulSoup nos deja navegar esa estructura y buscar elementos por su etiqueta, clase CSS, o atributos.

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

print("Tipo:", type(soup))
print("Título de la página:", soup.title.text)

Tipo: <class 'bs4.BeautifulSoup'>
Título de la página: 
    All products | Books to Scrape - Sandbox



Paso 4 — Encontrar las categorías
Abrí el sitio en el navegador: http://books.toscrape.com/
Hacé clic derecho sobre cualquier categoría del sidebar izquierdo → Inspeccionar. Vas a ver algo así:
<ul class="nav nav-list">
    <li>
        <a href="catalogue/category/books_1/index.html">Books</a>
        <ul>
            <li><a href="catalogue/category/books/travel_2/index.html">Travel</a></li>
            <li><a href="catalogue/category/books/mystery_3/index.html">Mystery</a></li>
            ...
        </ul>
    </li>
</ul>
¿Qué hace soup.select()?
Usa selectores CSS igual que en el navegador. ul.nav.nav-list li a significa: buscá todos los <a> que estén dentro de un <li> que esté dentro de un <ul> con clases nav y nav-list.

In [5]:
# Buscamos todos los <a> dentro del nav de categorías
# [1:] descarta el primero que es "Books" (categoría general, no nos interesa)
categorias = soup.select("ul.nav.nav-list li a")[1:]

print(f"Total de categorías: {len(categorias)}")
print("\nPrimeras 5:")
for cat in categorias[:5]:
    print(" -", cat.text.strip(), "→", cat["href"])

Total de categorías: 50

Primeras 5:
 - Travel → catalogue/category/books/travel_2/index.html
 - Mystery → catalogue/category/books/mystery_3/index.html
 - Historical Fiction → catalogue/category/books/historical-fiction_4/index.html
 - Sequential Art → catalogue/category/books/sequential-art_5/index.html
 - Classics → catalogue/category/books/classics_6/index.html


Paso 5 — Construir las URLs completas
Fijate que el href que nos devuelve es relativo: catalogue/category/books/travel_2/index.html
Para poder visitarlo necesitamos la URL completa: http://books.toscrape.com/catalogue/category/books/travel_2/index.html
En una celda nueva:
¿Qué hace urljoin?
Combina una URL base con una relativa y resuelve la ruta correcta automáticamente.
Es más seguro que concatenar strings con + porque maneja correctamente las barras y rutas relativas como ../.

In [6]:
from urllib.parse import urljoin

BASE_URL = "http://books.toscrape.com/"

# Probamos con la primera categoría
primera = categorias[0]

nombre = primera.text.strip()
url_completa = urljoin(BASE_URL, primera["href"])

print("Nombre:", nombre)
print("URL completa:", url_completa)

Nombre: Travel
URL completa: http://books.toscrape.com/catalogue/category/books/travel_2/index.html


Paso 6 — Entrar a una categoría y ver los libros
Ahora vamos a entrar a esa URL y buscar los libros que tiene. En una celda nueva:
¿Por qué miramos el HTML del primer libro?
Antes de extraer datos necesitamos entender la estructura. El .prettify() nos muestra el HTML indentado para leerlo más fácil. Con eso vamos a saber exactamente dónde está el título, precio y rating.

¿Por qué miramos el HTML del primer libro?
Antes de extraer datos necesitamos entender la estructura. El .prettify() nos muestra el HTML indentado para leerlo más fácil. Con eso vamos a saber exactamente dónde está el título, precio y rating.

Perfecto. ✅ Vemos la estructura del libro claramente.
Del HTML podemos identificar exactamente dónde está cada dato:

Título → <a href="..." > dentro del <h3> — tiene el atributo title con el nombre completo
Rating → <p class="star-rating Two"> — la segunda clase es el rating en palabras
Precio → todavía no se ve, está más abajo en el HTML

In [7]:
# Entramos a la categoría Travel
respuesta_cat = requests.get(url_completa)
soup_cat = BeautifulSoup(respuesta_cat.text, "html.parser")

# Cada libro está dentro de un <article class="product_pod">
libros = soup_cat.select("article.product_pod")

print(f"Libros encontrados en Travel: {len(libros)}")
print("\nPrimer libro (HTML resumido):")
print(libros[0].prettify()[:500])

Libros encontrados en Travel: 11

Primer libro (HTML resumido):
<article class="product_pod">
 <div class="image_container">
  <a href="../../../its-only-the-himalayas_981/index.html">
   <img alt="It's Only the Himalayas" class="thumbnail" src="../../../../media/cache/27/a5/27a53d0bb95bdd88288eaf66c9230d7e.jpg"/>
  </a>
 </div>
 <p class="star-rating Two">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="../../../its-only-the-hima


Paso 7 — Extraer datos del primer libro
En una celda nueva:

In [8]:
# Tomamos el primer libro para practicar
libro = libros[0]

# ── Título ──────────────────────────────────────────
titulo = libro.select_one("h3 a")["title"]

# ── Rating ──────────────────────────────────────────
# <p class="star-rating Two"> → ["star-rating", "Two"] → tomamos índice 1
rating_texto = libro.select_one("p.star-rating")["class"][1]

# Diccionario para convertir texto a número
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
rating = RATING_MAP[rating_texto]

# ── Precio ──────────────────────────────────────────
precio_texto = libro.select_one("p.price_color").text
precio = float(precio_texto.replace("£", "").replace("Â", "").strip())

print("Título:", titulo)
print("Rating:", rating, "estrellas")
print("Precio: £", precio)

Título: It's Only the Himalayas
Rating: 2 estrellas
Precio: £ 45.17


Paso 8 — Extraer todos los libros de la categoría
Ahora que sabemos cómo extraer datos de un libro, lo aplicamos a todos con un loop.
En una celda nueva:
Fijate algo interesante — algunos títulos tienen caracteres raros como Noahâs. Eso es un problema de encoding del sitio que vamos a ignorar por ahora, no afecta el funcionamiento.

In [9]:
libros_travel = []

for libro in libros:
    titulo = libro.select_one("h3 a")["title"]
    rating_texto = libro.select_one("p.star-rating")["class"][1]
    rating = RATING_MAP[rating_texto]
    precio_texto = libro.select_one("p.price_color").text
    precio = float(precio_texto.replace("£", "").replace("Â", "").strip())
    
    libros_travel.append({
        "titulo": titulo,
        "rating": rating,
        "precio": precio
    })

print(f"Total libros extraídos: {len(libros_travel)}")
print("\nTodos los libros:")
for l in libros_travel:
    print(f"  [{l['rating']}★] £{l['precio']:.2f} — {l['titulo']}")

Total libros extraídos: 11

Todos los libros:
  [2★] £45.17 — It's Only the Himalayas
  [4★] £49.43 — Full Moon over Noahâs Ark: An Odyssey to Mount Ararat and Beyond
  [3★] £48.87 — See America: A Celebration of Our National Parks & Treasured Sites
  [2★] £36.94 — Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel
  [3★] £37.33 — Under the Tuscan Sun
  [2★] £44.34 — A Summer In Europe
  [1★] £30.54 — The Great Railway Bazaar
  [4★] £56.88 — A Year in Provence (Provence #1)
  [1★] £23.21 — The Road to Little Dribbling: Adventures of an American in Britain (Notes From a Small Island #2)
  [3★] £38.95 — Neither Here nor There: Travels in Europe
  [5★] £26.08 — 1,000 Places to See Before You Die


Paso 9 — El problema de la paginación
Travel tiene solo 11 libros y caben en una página. Pero otras categorías tienen más libros y se dividen en múltiples páginas.
Mirá la URL de la segunda página de cualquier categoría grande:
http://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
El sitio tiene un botón "next" cuando hay más páginas. En una celda nueva:

In [10]:
# ¿Tiene Travel botón next?
next_btn = soup_cat.select_one("li.next a")
print("Botón next en Travel:", next_btn)

# Probemos con una categoría más grande: Mystery
url_mystery = urljoin(BASE_URL, "catalogue/category/books/mystery_3/index.html")
soup_mystery = BeautifulSoup(requests.get(url_mystery).text, "html.parser")

next_btn_mystery = soup_mystery.select_one("li.next a")
print("Botón next en Mystery:", next_btn_mystery)

if next_btn_mystery:
    print("URL siguiente página:", urljoin(url_mystery, next_btn_mystery["href"]))

Botón next en Travel: None
Botón next en Mystery: <a href="page-2.html">next</a>
URL siguiente página: http://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html


Paso 10 — Función para scrapear una categoría completa con paginación
Ahora convertimos todo lo que aprendimos en una función reutilizable. En una celda nueva:

In [11]:
import time

def scrapear_categoria(url_categoria):
    """
    Extrae todos los libros de una categoría manejando la paginación.
    Sigue el botón 'next' hasta que no haya más páginas.
    """
    libros = []
    url_actual = url_categoria
    pagina = 1

    while url_actual:  # mientras haya página siguiente
        print(f"  Scrapeando página {pagina}...")
        
        respuesta = requests.get(url_actual)
        soup = BeautifulSoup(respuesta.text, "html.parser")
        
        # Extraer libros de esta página
        articulos = soup.select("article.product_pod")
        
        for libro in articulos:
            titulo = libro.select_one("h3 a")["title"]
            rating_texto = libro.select_one("p.star-rating")["class"][1]
            rating = RATING_MAP[rating_texto]
            precio_texto = libro.select_one("p.price_color").text
            precio = float(precio_texto.replace("£", "").replace("Â", "").strip())
            
            libros.append({
                "titulo": titulo,
                "rating": rating,
                "precio": precio
            })
        
        # ¿Hay página siguiente?
        next_btn = soup.select_one("li.next a")
        if next_btn:
            url_actual = urljoin(url_actual, next_btn["href"])
            pagina += 1
        else:
            url_actual = None  # no hay más páginas, salir del while
        
        time.sleep(0.5)  # pausa entre páginas, scraping ético
    
    return libros


# Probamos con Mystery
print("Scrapeando Mystery...")
libros_mystery = scrapear_categoria(url_mystery)
print(f"\nTotal libros en Mystery: {len(libros_mystery)}")
print("\nPrimeros 3:")
for l in libros_mystery[:3]:
    print(f"  [{l['rating']}★] £{l['precio']:.2f} — {l['titulo']}")

Scrapeando Mystery...
  Scrapeando página 1...
  Scrapeando página 2...

Total libros en Mystery: 32

Primeros 3:
  [4★] £47.82 — Sharp Objects
  [1★] £19.63 — In a Dark, Dark Wood
  [4★] £56.50 — The Past Never Ends


Paso 11 — Scrapear TODAS las categorías
Ahora combinamos todo: la lista de categorías del paso 5 y la función del paso 10.
En una celda nueva:

In [12]:
todos_los_libros = []

print(f"Scrapeando {len(categorias)} categorías...\n")

for i, cat in enumerate(categorias):
    nombre_cat = cat.text.strip()
    url_cat = urljoin(BASE_URL, cat["href"])
    
    print(f"[{i+1}/{len(categorias)}] {nombre_cat}")
    
    libros_cat = scrapear_categoria(url_cat)
    
    # Agregamos el nombre de categoría a cada libro
    for libro in libros_cat:
        libro["categoria"] = nombre_cat
    
    todos_los_libros.extend(libros_cat)
    print(f"  → {len(libros_cat)} libros\n")

print(f"✅ Scraping completo")
print(f"   Total libros: {len(todos_los_libros)}")
print(f"   Total categorías: {len(categorias)}")

Scrapeando 50 categorías...

[1/50] Travel
  Scrapeando página 1...
  → 11 libros

[2/50] Mystery
  Scrapeando página 1...
  Scrapeando página 2...
  → 32 libros

[3/50] Historical Fiction
  Scrapeando página 1...
  Scrapeando página 2...
  → 26 libros

[4/50] Sequential Art
  Scrapeando página 1...
  Scrapeando página 2...
  Scrapeando página 3...
  Scrapeando página 4...
  → 75 libros

[5/50] Classics
  Scrapeando página 1...
  → 19 libros

[6/50] Philosophy
  Scrapeando página 1...
  → 11 libros

[7/50] Romance
  Scrapeando página 1...
  Scrapeando página 2...
  → 35 libros

[8/50] Womens Fiction
  Scrapeando página 1...
  → 17 libros

[9/50] Fiction
  Scrapeando página 1...
  Scrapeando página 2...
  Scrapeando página 3...
  Scrapeando página 4...
  → 65 libros

[10/50] Childrens
  Scrapeando página 1...
  Scrapeando página 2...
  → 29 libros

[11/50] Religion
  Scrapeando página 1...
  → 7 libros

[12/50] Nonfiction
  Scrapeando página 1...
  Scrapeando página 2...
  Scrapeando pá

Momento de reflexión — ¿qué construiste hasta acá?
requests.get(url)           → petición HTTP al servidor
BeautifulSoup(html, parser) → parsear el HTML en árbol navegable
soup.select("selector css") → buscar elementos por clase/etiqueta
urljoin(base, relativa)     → construir URLs absolutas
while url_actual:           → paginación automática
time.sleep(0.5)             → scraping ético

Perfecto. Vamos a crear las tablas.
En una celda nueva:

In [13]:
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

# SQLite no activa las foreign keys por defecto, hay que habilitarlas
cursor.execute("PRAGMA foreign_keys = ON")

print("✅ Conexión establecida")

✅ Conexión establecida


Perfecto. Ahora creamos las tablas una por una para que entiendas cada decisión.
Primero categories — la creamos primero porque books depende de ella:
¿Qué significa cada parte?

CREATE TABLE IF NOT EXISTS → si la tabla ya existe no falla, simplemente la ignora
PRIMARY KEY AUTOINCREMENT → SQLite genera el ID automáticamente: 1, 2, 3...
NOT NULL → ese campo no puede quedar vacío
UNIQUE → no pueden existir dos categorías con el mismo nombre o slug
conn.commit() → confirma los cambios en el archivo. Sin esto los cambios se pierden

In [14]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        name       TEXT    NOT NULL UNIQUE,
        slug       TEXT    NOT NULL UNIQUE
    )
""")

conn.commit()
print("✅ Tabla categories creada")

✅ Tabla categories creada


Ahora authors — la creamos antes que books porque book_author depende de ambas:
¿Qué es nuevo acá?

Los campos sin NOT NULL → pueden ser NULL. Eso es intencional porque estos datos vienen de la API y puede que no los encontremos
DEFAULT 'pending' → si no especificamos un valor para api_status, SQLite pone 'pending' automáticamente
DEFAULT CURRENT_TIMESTAMP → guarda automáticamente la fecha y hora en que se insertó el registro

In [15]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS authors (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        name              TEXT    NOT NULL UNIQUE,
        birth_year        INTEGER,
        country           TEXT,
        external_api_id   TEXT,
        total_known_works INTEGER,
        api_source        TEXT,
        api_status        TEXT DEFAULT 'pending',
        created_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

conn.commit()
print("✅ Tabla authors creada")

✅ Tabla authors creada


Ahora books:
¿Qué es nuevo acá?

REAL → tipo de dato para números decimales (el precio tiene centavos)
CHECK (rating BETWEEN 1 AND 5) → validación a nivel de base de datos. Si intentás insertar un rating de 6, SQLite rechaza el insert con error
REFERENCES categories(id) → esta es la foreign key. Le dice a SQLite que category_id debe existir en la tabla categories. No podés insertar un libro con una categoría que no existe

In [16]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        title       TEXT    NOT NULL,
        price       REAL    NOT NULL,
        rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
        category_id INTEGER NOT NULL REFERENCES categories(id),
        url         TEXT,
        description TEXT
    )
""")

conn.commit()
print("✅ Tabla books creada")

✅ Tabla books creada


Ahora la última tabla, book_author:
¿Qué es nuevo acá?

PRIMARY KEY (book_id, author_id) → clave primaria compuesta. En vez de un solo campo como ID, la combinación de los dos campos es única. Esto evita que el mismo par libro-autor se inserte dos veces
ON DELETE CASCADE → si borrás un libro de la tabla books, todas sus filas en book_author se borran automáticamente. Evita registros huérfanos (filas que apuntan a un libro que ya no existe)

In [17]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS book_author (
        book_id   INTEGER NOT NULL REFERENCES books(id)   ON DELETE CASCADE,
        author_id INTEGER NOT NULL REFERENCES authors(id) ON DELETE CASCADE,
        PRIMARY KEY (book_id, author_id)
    )
""")

conn.commit()
print("✅ Tabla book_author creada")

✅ Tabla book_author creada


Perfecto. Las 4 tablas están creadas. Vamos a verificar que todo quedó bien:

In [18]:
# Verificar que las tablas existen
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tablas = cursor.fetchall()

print("Tablas en la base de datos:")
for tabla in tablas:
    print(f"  ✅ {tabla[0]}")

Tablas en la base de datos:
  ✅ categories
  ✅ sqlite_sequence
  ✅ authors
  ✅ books
  ✅ book_author


Siguiente paso: insertar las categorías en la DB
En una celda nueva:

In [19]:
def insertar_categoria(nombre, slug):
    """
    Inserta una categoría y retorna su ID.
    INSERT OR IGNORE: si ya existe no falla, simplemente la ignora.
    """
    cursor.execute(
        "INSERT OR IGNORE INTO categories (name, slug) VALUES (?, ?)",
        (nombre, slug)
    )
    conn.commit()
    
    # Recuperamos el ID (sea nuevo o ya existente)
    cursor.execute("SELECT id FROM categories WHERE slug = ?", (slug,))
    return cursor.fetchone()[0]

# Probamos con Travel
id_travel = insertar_categoria("Travel", "travel_2")
print(f"✅ Travel insertada con ID: {id_travel}")

✅ Travel insertada con ID: 1


Ahora insertamos todas las categorías que scrapeamos:

In [20]:
def extraer_slug(href):
    """
    Extrae el slug de la URL de la categoría.
    Ejemplo: "catalogue/category/books/mystery_3/index.html" → "mystery_3"
    """
    return href.split("/")[-2]

# Insertar todas las categorías
print("Insertando categorías...")

for cat in categorias:
    nombre = cat.text.strip()
    slug   = extraer_slug(cat["href"])
    id_cat = insertar_categoria(nombre, slug)
    print(f"  ✅ {nombre} → ID: {id_cat}")

Insertando categorías...
  ✅ Travel → ID: 1
  ✅ Mystery → ID: 3
  ✅ Historical Fiction → ID: 4
  ✅ Sequential Art → ID: 5
  ✅ Classics → ID: 6
  ✅ Philosophy → ID: 7
  ✅ Romance → ID: 8
  ✅ Womens Fiction → ID: 9
  ✅ Fiction → ID: 10
  ✅ Childrens → ID: 11
  ✅ Religion → ID: 12
  ✅ Nonfiction → ID: 13
  ✅ Music → ID: 14
  ✅ Default → ID: 15
  ✅ Science Fiction → ID: 16
  ✅ Sports and Games → ID: 17
  ✅ Add a comment → ID: 18
  ✅ Fantasy → ID: 19
  ✅ New Adult → ID: 20
  ✅ Young Adult → ID: 21
  ✅ Science → ID: 22
  ✅ Poetry → ID: 23
  ✅ Paranormal → ID: 24
  ✅ Art → ID: 25
  ✅ Psychology → ID: 26
  ✅ Autobiography → ID: 27
  ✅ Parenting → ID: 28
  ✅ Adult Fiction → ID: 29
  ✅ Humor → ID: 30
  ✅ Horror → ID: 31
  ✅ History → ID: 32
  ✅ Food and Drink → ID: 33
  ✅ Christian Fiction → ID: 34
  ✅ Business → ID: 35
  ✅ Biography → ID: 36
  ✅ Thriller → ID: 37
  ✅ Contemporary → ID: 38
  ✅ Spirituality → ID: 39
  ✅ Academic → ID: 40
  ✅ Self Help → ID: 41
  ✅ Historical → ID: 42
  ✅ Christia

Ahora insertamos los libros:

In [21]:
def insertar_libro(titulo, precio, rating, category_id, url=""):
    """
    Inserta un libro y retorna su ID.
    """
    cursor.execute("""
        INSERT INTO books (title, price, rating, category_id, url)
        VALUES (?, ?, ?, ?, ?)
    """, (titulo, precio, rating, category_id, url))
    conn.commit()
    return cursor.lastrowid

# Insertar todos los libros
print("Insertando libros...")

for libro in todos_los_libros:
    # Necesitamos el ID de la categoría del libro
    cursor.execute(
        "SELECT id FROM categories WHERE name = ?",
        (libro["categoria"],)
    )
    resultado = cursor.fetchone()
    
    if resultado:
        category_id = resultado[0]
        insertar_libro(
            libro["titulo"],
            libro["precio"],
            libro["rating"],
            category_id
        )

# Verificar
cursor.execute("SELECT COUNT(*) FROM books")
total = cursor.fetchone()[0]
print(f"✅ Libros insertados: {total}")

Insertando libros...
✅ Libros insertados: 1000


Vamos a verificar que todo quedó bien con una consulta rápida:

In [23]:
# Ver una muestra de libros con su categoría
cursor.execute("""
    SELECT b.title, b.price, b.rating, c.name
    FROM books b
    JOIN categories c ON b.category_id = c.id
    LIMIT 5
""")

resultados = cursor.fetchall()

print("Muestra de libros en la DB:")
print("-" * 65)
for row in resultados:
    print(f"  [{row[2]}★] £{row[1]:.2f} — {row[0][:40]} ({row[3]})")

Muestra de libros en la DB:
-----------------------------------------------------------------
  [2★] £45.17 — It's Only the Himalayas (Travel)
  [4★] £49.43 — Full Moon over Noahâs Ark: An Odyssey  (Travel)
  [3★] £48.87 — See America: A Celebration of Our Nation (Travel)
  [2★] £36.94 — Vagabonding: An Uncommon Guide to the Ar (Travel)
  [3★] £37.33 — Under the Tuscan Sun (Travel)


Siguiente etapa: la API de Open Library
Vamos a buscar el autor de cada libro por título. En una celda nueva primero probamos con un solo libro para entender la respuesta de la API:

El endpoint que iba a usar (/search.json por título) nos da el autor pero no nos da los datos del autor — año de nacimiento, país, cantidad de obras. Para eso necesitamos el endpoint específico de autores.
El flujo correcto es en dos pasos:
1. Buscar libro por título → nos devuelve el nombre del autor
2. Buscar autor por nombre → nos devuelve birth_year, country, works, etc.
Ambos endpoints de Open Library:
Paso 1: https://openlibrary.org/search.json?title=Sharp+Objects&limit=1
Paso 2: https://openlibrary.org/search/authors.json?q=Gillian+Flynn&limit=1

Antes de escribir código, ¿entendés por qué necesitamos dos llamadas y no una sola? Quiero asegurarme de que cuando te pregunten en la presentación puedas explicarlo.

In [24]:
 # PASO 1: buscar libro por título para obtener el nombre del autor
titulo_prueba = "Sharp Objects"

url_buscar_libro = "https://openlibrary.org/search.json"
params = {"title": titulo_prueba, "limit": 1}

respuesta = requests.get(url_buscar_libro, params=params, timeout=10)
data = respuesta.json()

print(f"Status: {respuesta.status_code}")
print(f"Resultados encontrados: {data['numFound']}")

if data['numFound'] > 0:
    doc = data['docs'][0]
    print(f"\nTítulo:  {doc.get('title')}")
    print(f"Autor:   {doc.get('author_name')}")
    print(f"Año pub: {doc.get('first_publish_year')}")

Status: 200
Resultados encontrados: 46

Título:  Sharp Objects
Autor:   ['Gillian Flynn']
Año pub: 2006


In [25]:
# PASO 2: buscar datos del autor por nombre
autor_nombre = data['docs'][0].get('author_name')[0]  # "Gillian Flynn"

url_buscar_autor = "https://openlibrary.org/search/authors.json"
params_autor = {"q": autor_nombre, "limit": 1}

respuesta_autor = requests.get(url_buscar_autor, params=params_autor, timeout=10)
data_autor = respuesta_autor.json()

print(f"Status: {respuesta_autor.status_code}")
print(f"Resultados: {data_autor['numFound']}")

if data_autor['numFound'] > 0:
    doc_autor = data_autor['docs'][0]
    print(f"\nNombre:      {doc_autor.get('name')}")
    print(f"ID externo:  {doc_autor.get('key')}")
    print(f"Nacimiento:  {doc_autor.get('birth_date')}")
    print(f"País:        {doc_autor.get('top_subjects')}")
    print(f"Obras:       {doc_autor.get('work_count')}")

Status: 200
Resultados: 5

Nombre:      Gillian Flynn
ID externo:  OL1433006A
Nacimiento:  1971-02-24
País:        ['Fiction', 'Fiction, suspense', 'Fiction, thrillers, suspense', 'New York Times bestseller', 'Missouri', 'Crimes against', 'Women journalists, fiction', 'Thrillers', 'Missouri, fiction', 'Large type books']
Obras:       41


In [26]:
# Buscar detalle del autor usando su ID
autor_id = doc_autor.get('key')  # "OL1433006A"

url_detalle = f"https://openlibrary.org/authors/{autor_id}.json"
respuesta_detalle = requests.get(url_detalle, timeout=10)
data_detalle = respuesta_detalle.json()

print(f"Status: {respuesta_detalle.status_code}")
print(f"\nNombre:       {data_detalle.get('name')}")
print(f"Nacimiento:   {data_detalle.get('birth_date')}")
print(f"Birth place:  {data_detalle.get('birth_place')}")
print(f"Bio:          {str(data_detalle.get('bio', ''))[:100]}")

Status: 200

Nombre:       Gillian Flynn
Nacimiento:   1971-02-24
Birth place:  None
Bio:          {'type': '/type/text', 'value': "Flynn, who lives in Chicago, grew up in Kansas City, Missouri. She 


In [27]:
# Probamos con un autor más clásico
url_buscar_autor = "https://openlibrary.org/search/authors.json"
params_autor = {"q": "George Orwell", "limit": 1}

respuesta = requests.get(url_buscar_autor, params=params_autor, timeout=10)
doc = respuesta.json()['docs'][0]

autor_id = doc.get('key')
url_detalle = f"https://openlibrary.org/authors/{autor_id}.json"
detalle = requests.get(url_detalle, timeout=10).json()

print(f"Nombre:      {detalle.get('name')}")
print(f"Nacimiento:  {detalle.get('birth_date')}")
print(f"Birth place: {detalle.get('birth_place')}")
print(f"Obras:       {doc.get('work_count')}")

Nombre:      George Orwell
Nacimiento:  25 June 1903
Birth place: None
Obras:       683


In [32]:
url_buscar_autor = "https://openlibrary.org/search/authors.json"
params = {"q": "Jane Austen", "limit": 1}

respuesta = requests.get(url_buscar_autor, params=params, timeout=10)
doc = respuesta.json()['docs'][0]

autor_id = doc.get('key')
url_detalle = f"https://openlibrary.org/authors/{autor_id}.json"
detalle = requests.get(url_detalle, timeout=10).json()

print(f"Nombre:      {detalle.get('name')}")
print(f"Nacimiento:  {detalle.get('birth_date')}")
print(f"Birth place: {detalle.get('birth_place')}")
print(f"Obras:       {doc.get('work_count')}")

Nombre:      Jane Austen
Nacimiento:  December 16, 1775
Birth place: None
Obras:       2209


In [22]:
# Verificar estado completo de la base de datos
cursor.execute("SELECT COUNT(*) FROM books")
libros = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM categories")
categorias_db = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM authors")
autores = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM book_author")
relaciones = cursor.fetchone()[0]

print("Estado de la base de datos:")
print(f"  📚 Libros:      {libros}")
print(f"  📂 Categorías:  {categorias_db}")
print(f"  ✍️  Autores:     {autores}")
print(f"  🔗 Relaciones:  {relaciones}")

Estado de la base de datos:
  📚 Libros:      1000
  📂 Categorías:  50
  ✍️  Autores:     0
  🔗 Relaciones:  0


In [24]:
def buscar_autor_openlibrary(nombre_autor):
    """
    Busca un autor en Open Library y retorna sus datos básicos.
    """
    url = "https://openlibrary.org/search/authors.json"
    params = {"q": nombre_autor, "limit": 1}
    
    respuesta = requests.get(url, params=params, timeout=10)
    data = respuesta.json()
    
    if data['numFound'] == 0:
        return None
    
    doc = data['docs'][0]
    
    # Limpiar el año: "1971-02-24" → 1971 / "25 June 1903" → 1903
    birth_year_raw = doc.get("birth_date")
    birth_year = None
    if birth_year_raw:
        # Buscamos 4 dígitos consecutivos en el string
        import re
        años = re.findall(r'\d{4}', str(birth_year_raw))
        birth_year = int(años[0]) if años else None
    
    return {
        "external_api_id":   doc.get("key"),
        "birth_year":        birth_year,
        "total_known_works": doc.get("work_count"),
        "api_source":        "openlibrary+wikipedia"
    }

# Probamos
resultado = buscar_autor_openlibrary("Gillian Flynn")
print(resultado)

{'external_api_id': 'OL1433006A', 'birth_year': 1971, 'total_known_works': 41, 'api_source': 'openlibrary+wikipedia'}


In [25]:
def buscar_pais_wikipedia(nombre_autor):
    """
    Busca la nacionalidad de un autor en Wikipedia.
    Retorna la primera palabra de la descripción.
    """
    headers_wiki = {
        "User-Agent": "TH-Challenge4/1.0 (educational project)"
    }
    
    # Wikipedia necesita el nombre con guiones bajos en la URL
    nombre_url = nombre_autor.replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{nombre_url}"
    
    respuesta = requests.get(url, headers=headers_wiki, timeout=10)
    
    if respuesta.status_code != 200:
        return None
    
    data = respuesta.json()
    descripcion = data.get('description', None)
    
    if not descripcion:
        return None
    
    # Primera palabra → nacionalidad
    return descripcion.split()[0]

# Probamos
pais = buscar_pais_wikipedia("Gillian Flynn")
print("País:", pais)

País: American


In [26]:
def obtener_datos_autor(nombre_autor):
    """
    Combina Open Library y Wikipedia para obtener
    todos los datos necesarios de un autor.
    """
    # Datos de Open Library
    datos_ol = buscar_autor_openlibrary(nombre_autor)
    
    # Datos de Wikipedia
    pais = buscar_pais_wikipedia(nombre_autor)
    
    if datos_ol is None:
        return {
            "external_api_id":   None,
            "birth_year":        None,
            "total_known_works": None,
            "country":           pais,
            "api_source":        "openlibrary+wikipedia",
            "api_status":        "not_found"
        }
    
    datos_ol["country"]    = pais
    datos_ol["api_status"] = "found"
    
    return datos_ol

# Probamos
datos = obtener_datos_autor("Gillian Flynn")
print(datos)

{'external_api_id': 'OL1433006A', 'birth_year': 1971, 'total_known_works': 41, 'api_source': 'openlibrary+wikipedia', 'country': 'American', 'api_status': 'found'}


In [27]:
# Endpoint detallado de Open Library para Gillian Flynn
url = "https://openlibrary.org/authors/OL1433006A.json"
respuesta = requests.get(url, timeout=10)
data = respuesta.json()

# Ver TODAS las claves que devuelve
print("Campos disponibles:")
for clave, valor in data.items():
    print(f"  {clave}: {valor}")

Campos disponibles:
  name: Gillian Flynn
  bio: {'type': '/type/text', 'value': "Flynn, who lives in Chicago, grew up in Kansas City, Missouri. She graduated at the University of Kansas, and qualified for a Master's degree from Northwestern University."}
  source_records: ['amazon:1101902884', 'bwb:9788417125936', 'amazon:8496929051', 'amazon:1474620736']
  remote_ids: {'isni': '0000000114791438', 'viaf': '103168327', 'wikidata': 'Q311755', 'goodreads': '2383', 'imdb': 'nm5058839', 'lc_naf': 'n2005086885', 'opac_sbn': 'RAVV545007'}
  alternate_names: ['GILLIAN FLYNN']
  photos: [7260938, -1]
  personal_name: Gillian Flynn
  type: {'key': '/type/author'}
  birth_date: 1971-02-24
  links: [{'title': 'GillianFlynn.com', 'url': 'http://gillian-flynn.com/', 'type': {'key': '/type/link'}}]
  key: /authors/OL1433006A
  latest_revision: 16
  revision: 16
  created: {'type': '/type/datetime', 'value': '2008-04-01T03:28:50.625462'}
  last_modified: {'type': '/type/datetime', 'value': '2025-07-3

In [28]:
headers_wiki = {
    "User-Agent": "TH-Challenge4/1.0 (educational project)"
}

url = "https://en.wikipedia.org/api/rest_v1/page/summary/Gillian_Flynn"
respuesta = requests.get(url, headers=headers_wiki, timeout=10)
data = respuesta.json()

print("Campos disponibles:")
for clave, valor in data.items():
    print(f"  {clave}: {str(valor)[:100]}")

Campos disponibles:
  type: standard
  title: Gillian Flynn
  displaytitle: <span lang="en" dir="ltr"><span class="mw-page-title-main">Gillian Flynn</span></span>
  namespace: {'id': 0, 'text': ''}
  wikibase_item: Q311755
  titles: {'canonical': 'Gillian_Flynn', 'normalized': 'Gillian Flynn', 'display': '<span lang="en" dir="ltr">
  pageid: 12513388
  thumbnail: {'source': 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a5/Gillian_Flynn_2014_%28cropped%
  originalimage: {'source': 'https://upload.wikimedia.org/wikipedia/commons/a/a5/Gillian_Flynn_2014_%28cropped%29.jpg
  lang: en
  dir: ltr
  revision: 1347280549
  tid: 022baf47-312a-11f1-8764-07c07dcdb395
  timestamp: 2026-04-05T19:59:52Z
  description: American writer (born 1971)
  description_source: local
  content_urls: {'desktop': {'page': 'https://en.wikipedia.org/wiki/Gillian_Flynn', 'revisions': 'https://en.wikiped
  extract: Gillian Schieber Flynn is an American author, screenwriter, and producer, best known for her 

Perfecto. Antes de escribir código, definamos exactamente qué saca cada API:
Open Library → external_api_id, total_known_works
Wikipedia    → country (primera palabra), birth_year (número entre paréntesis)

In [29]:
def buscar_pais_wikipedia(nombre_autor):
    """
    Busca nacionalidad y año de nacimiento de un autor en Wikipedia.
    Retorna un dict con country y birth_year.
    """
    import re
    
    headers_wiki = {
        "User-Agent": "TH-Challenge4/1.0 (educational project)"
    }
    
    nombre_url = nombre_autor.replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{nombre_url}"
    
    respuesta = requests.get(url, headers=headers_wiki, timeout=10)
    
    if respuesta.status_code != 200:
        return {"country": None, "birth_year": None}
    
    descripcion = respuesta.json().get('description', None)
    
    if not descripcion:
        return {"country": None, "birth_year": None}
    
    # País → primera palabra
    country = descripcion.split()[0]
    
    # Año → 4 dígitos entre paréntesis
    años = re.findall(r'\d{4}', descripcion)
    birth_year = int(años[0]) if años else None
    
    return {"country": country, "birth_year": birth_year}

# Probamos
resultado = buscar_pais_wikipedia("Gillian Flynn")
print(resultado)

{'country': 'American', 'birth_year': 1971}


In [30]:
def buscar_autor_openlibrary(nombre_autor):
    """
    Busca un autor en Open Library.
    Retorna external_api_id y total_known_works.
    """
    url = "https://openlibrary.org/search/authors.json"
    params = {"q": nombre_autor, "limit": 1}
    
    respuesta = requests.get(url, params=params, timeout=10)
    data = respuesta.json()
    
    if data['numFound'] == 0:
        return {"external_api_id": None, "total_known_works": None}
    
    doc = data['docs'][0]
    
    return {
        "external_api_id":   doc.get("key"),
        "total_known_works": doc.get("work_count")
    }

# Probamos
resultado = buscar_autor_openlibrary("Gillian Flynn")
print(resultado)

{'external_api_id': 'OL1433006A', 'total_known_works': 41}


Perfecto. ✅ Ahora combinamos las dos en la función final:


In [31]:
def obtener_datos_autor(nombre_autor):
    """
    Combina Open Library y Wikipedia para obtener
    todos los datos necesarios de un autor.
    """
    datos_ol   = buscar_autor_openlibrary(nombre_autor)
    datos_wiki = buscar_pais_wikipedia(nombre_autor)
    
    # Determinar api_status
    if datos_ol["external_api_id"] is None and datos_wiki["country"] is None:
        api_status = "not_found"
    else:
        api_status = "found"
    
    return {
        "external_api_id":   datos_ol["external_api_id"],
        "total_known_works": datos_ol["total_known_works"],
        "country":           datos_wiki["country"],
        "birth_year":        datos_wiki["birth_year"],
        "api_source":        "openlibrary+wikipedia",
        "api_status":        api_status
    }

# Probamos
datos = obtener_datos_autor("Gillian Flynn")
print(datos)

{'external_api_id': 'OL1433006A', 'total_known_works': 41, 'country': 'American', 'birth_year': 1971, 'api_source': 'openlibrary+wikipedia', 'api_status': 'found'}


Ahora implementamos el cache antes de seguir. ¿Recordás para qué sirve?
El cache evita llamar a la API dos veces para el mismo autor. Si el mismo autor aparece en 10 libros → sin cache serían 20 llamadas (Open Library + Wikipedia × 10). Con cache → 2 llamadas y el resto se lee de memoria.

In [32]:
# Cache en memoria: dura mientras el notebook esté abierto
cache_autores = {}

def obtener_datos_autor_con_cache(nombre_autor):
    """
    Igual que obtener_datos_autor pero con cache.
    Si el autor ya fue consultado, retorna el resultado guardado.
    """
    # Si ya está en cache, retornar directamente sin llamar a la API
    if nombre_autor in cache_autores:
        print(f"  📦 Cache: {nombre_autor}")
        return cache_autores[nombre_autor]
    
    # Si no está en cache, consultar las APIs
    print(f"  🌐 API: {nombre_autor}")
    datos = obtener_datos_autor(nombre_autor)
    
    # Guardar en cache para la próxima vez
    cache_autores[nombre_autor] = datos
    
    return datos

# Probamos llamando dos veces al mismo autor
print("Primera llamada:")
obtener_datos_autor_con_cache("Gillian Flynn")
print("\nSegunda llamada:")
obtener_datos_autor_con_cache("Gillian Flynn")

Primera llamada:
  🌐 API: Gillian Flynn

Segunda llamada:
  📦 Cache: Gillian Flynn


{'external_api_id': 'OL1433006A',
 'total_known_works': 41,
 'country': 'American',
 'birth_year': 1971,
 'api_source': 'openlibrary+wikipedia',
 'api_status': 'found'}

In [33]:
# Traer todos los títulos de la base de datos
cursor.execute("SELECT id, title FROM books")
libros_db = cursor.fetchall()

print(f"Total libros: {len(libros_db)}")
print("\nPrimeros 5:")
for libro in libros_db[:5]:
    print(f"  ID: {libro[0]} — {libro[1]}")

Total libros: 1000

Primeros 5:
  ID: 1 — It's Only the Himalayas
  ID: 2 — Full Moon over Noahâs Ark: An Odyssey to Mount Ararat and Beyond
  ID: 3 — See America: A Celebration of Our National Parks & Treasured Sites
  ID: 4 — Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel
  ID: 5 — Under the Tuscan Sun


Ahora necesitamos una función que busque el autor de un libro por su título en Open Library:

In [34]:
def buscar_autor_por_titulo(titulo):
    """
    Busca el autor de un libro por su título en Open Library.
    Retorna el nombre del autor o None si no lo encuentra.
    """
    url = "https://openlibrary.org/search.json"
    params = {"title": titulo, "limit": 1}
    
    respuesta = requests.get(url, params=params, timeout=10)
    data = respuesta.json()
    
    if data['numFound'] == 0:
        return None
    
    autores = data['docs'][0].get('author_name')
    
    if not autores:
        return None
    
    # Retornamos el primer autor de la lista
    return autores[0]

# Probamos con un título conocido
autor = buscar_autor_por_titulo("Sharp Objects")
print("Autor encontrado:", autor)

Autor encontrado: Gillian Flynn


In [35]:
def insertar_autor(nombre, datos):
    """
    Inserta un autor en la DB y retorna su ID.
    INSERT OR IGNORE: si ya existe no duplica.
    """
    cursor.execute("""
        INSERT OR IGNORE INTO authors 
        (name, birth_year, country, external_api_id, total_known_works, api_source, api_status)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        nombre,
        datos["birth_year"],
        datos["country"],
        datos["external_api_id"],
        datos["total_known_works"],
        datos["api_source"],
        datos["api_status"]
    ))
    conn.commit()
    
    cursor.execute("SELECT id FROM authors WHERE name = ?", (nombre,))
    return cursor.fetchone()[0]


def vincular_libro_autor(book_id, author_id):
    """
    Crea la relación entre un libro y un autor en book_author.
    """
    cursor.execute(
        "INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)",
        (book_id, author_id)
    )
    conn.commit()

# Probamos con Gillian Flynn
datos = obtener_datos_autor_con_cache("Gillian Flynn")
author_id = insertar_autor("Gillian Flynn", datos)
print(f"✅ Autor insertado con ID: {author_id}")

  📦 Cache: Gillian Flynn
✅ Autor insertado con ID: 1


In [36]:
encontrados = 0
no_encontrados = 0
errores = 0

print(f"Procesando {len(libros_db)} libros...\n")

for i, (book_id, titulo) in enumerate(libros_db):
    
    # Progreso cada 50 libros
    if i % 50 == 0:
        print(f"  [{i}/{len(libros_db)}] procesando...")
    
    # Paso 1: buscar autor por título en Open Library
    nombre_autor = buscar_autor_por_titulo(titulo)
    
    if not nombre_autor:
        # No encontró autor → insertar desconocido
        datos = {
            "birth_year": None, "country": None,
            "external_api_id": None, "total_known_works": None,
            "api_source": "openlibrary+wikipedia", "api_status": "not_found"
        }
        nombre_autor = "Unknown"
        no_encontrados += 1
    else:
        # Paso 2: obtener datos completos del autor (con cache)
        datos = obtener_datos_autor_con_cache(nombre_autor)
        encontrados += 1
    
    # Insertar autor y vincular al libro
    author_id = insertar_autor(nombre_autor, datos)
    vincular_libro_autor(book_id, author_id)
    
    time.sleep(0.3)  # pausa entre requests

print(f"\n✅ Pipeline completo")
print(f"  Encontrados:    {encontrados}")
print(f"  No encontrados: {no_encontrados}")
print(f"  Errores:        {errores}")

Procesando 1000 libros...

  [0/1000] procesando...
  🌐 API: S. Bedford
  🌐 API: Frances Mayes
  🌐 API: Marilyn Brant
  🌐 API: Paul Theroux
  🌐 API: Patricia Schultz
  📦 Cache: Gillian Flynn
  🌐 API: Ruth Ware
  🌐 API: Jackson Burnett
  🌐 API: Julie McElwain
  🌐 API: Celeste Bradley
  🌐 API: Georgette Heyer
  🌐 API: Derek Landy


ConnectTimeout: HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /search.json?title=In+the+Woods+%28Dublin+Murder+Squad+%231%29&limit=1 (Caused by ConnectTimeoutError(<HTTPSConnection(host='openlibrary.org', port=443) at 0x2620f2597c0>, 'Connection to openlibrary.org timed out. (connect timeout=10)'))

In [37]:
# Test de conectividad
try:
    r = requests.get("https://openlibrary.org", timeout=5)
    print("Open Library:", r.status_code)
except Exception as e:
    print("Open Library: ❌", e)

try:
    r = requests.get("https://en.wikipedia.org", timeout=5, 
                     headers={"User-Agent": "TH-Challenge4/1.0"})
    print("Wikipedia:", r.status_code)
except Exception as e:
    print("Wikipedia: ❌", e)

Open Library: 200
Wikipedia: 200


In [38]:
def buscar_autor_por_titulo(titulo):
    """
    Busca el autor de un libro por su título en Open Library.
    Retorna el nombre del autor o None si no lo encuentra.
    """
    try:
        url = "https://openlibrary.org/search.json"
        params = {"title": titulo, "limit": 1}
        
        respuesta = requests.get(url, params=params, timeout=10)
        data = respuesta.json()
        
        if data['numFound'] == 0:
            return None
        
        autores = data['docs'][0].get('author_name')
        
        if not autores:
            return None
        
        return autores[0]
    
    except Exception as e:
        print(f"  ⚠️ Error con título: {titulo[:40]} → {e}")
        return None

# Probamos con el título problemático
autor = buscar_autor_por_titulo("In the Woods (Dublin Murder Squad #1)")
print("Autor:", autor)

Autor: None


In [39]:
import re

def limpiar_titulo(titulo):
    """
    Limpia el título para mejorar la búsqueda en la API.
    Elimina paréntesis y su contenido, y caracteres especiales.
    """
    # Eliminar contenido entre paréntesis: "(Dublin Murder Squad #1)" → ""
    titulo = re.sub(r'\(.*?\)', '', titulo)
    # Eliminar caracteres especiales
    titulo = titulo.strip()
    return titulo

# Probamos
titulo_original = "In the Woods (Dublin Murder Squad #1)"
titulo_limpio   = limpiar_titulo(titulo_original)

print(f"Original: {titulo_original}")
print(f"Limpio:   {titulo_limpio}")

autor = buscar_autor_por_titulo(titulo_limpio)
print(f"Autor:    {autor}")

Original: In the Woods (Dublin Murder Squad #1)
Limpio:   In the Woods
Autor:    Tana French


In [40]:
def buscar_autor_por_titulo(titulo):
    """
    Busca el autor de un libro por su título en Open Library.
    Limpia el título antes de buscar para mejorar los resultados.
    """
    try:
        # Limpiar título antes de buscar
        titulo_limpio = limpiar_titulo(titulo)
        
        url = "https://openlibrary.org/search.json"
        params = {"title": titulo_limpio, "limit": 1}
        
        respuesta = requests.get(url, params=params, timeout=10)
        data = respuesta.json()
        
        if data['numFound'] == 0:
            return None
        
        autores = data['docs'][0].get('author_name')
        
        if not autores:
            return None
        
        return autores[0]
    
    except Exception as e:
        print(f"  ⚠️ Error con título: {titulo[:40]} → {e}")
        return None

# Probamos con ambos títulos
print(buscar_autor_por_titulo("In the Woods (Dublin Murder Squad #1)"))
print(buscar_autor_por_titulo("Sharp Objects"))

Laura Ingalls Wilder
Gillian Flynn


In [41]:
# Probar si Open Library reconoce el UPC de Books to Scrape
upc = "a94350ee74deaa07"

url = f"https://openlibrary.org/isbn/{upc}.json"
respuesta = requests.get(url, timeout=10)

print(f"Status: {respuesta.status_code}")
print(f"Respuesta: {respuesta.text[:200]}")

Status: 404
Respuesta: 

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <meta name="format-detection" content="telephone=no">
    <meta name="viewport" content="width=device-width, initial-scale=1.0"


In [44]:
def buscar_autor_openlibrary(nombre_autor):
    try:
        url = "https://openlibrary.org/search/authors.json"
        params = {"q": nombre_autor, "limit": 1}
        respuesta = requests.get(url, params=params, timeout=10)
        data = respuesta.json()
        
        if data['numFound'] == 0:
            return {"external_api_id": None, "total_known_works": None}
        
        doc = data['docs'][0]
        return {
            "external_api_id":   doc.get("key"),
            "total_known_works": doc.get("work_count")
        }
    except Exception as e:
        print(f"  ⚠️ Error OpenLibrary autor: {nombre_autor[:30]} → {e}")
        return {"external_api_id": None, "total_known_works": None}


def buscar_autor_por_titulo(titulo):
    try:
        titulo_limpio = limpiar_titulo(titulo)
        if len(titulo_limpio) < 5:
            return None
        
        url = "https://openlibrary.org/search.json"
        params = {"title": titulo_limpio, "limit": 1}
        respuesta = requests.get(url, params=params, timeout=10)
        data = respuesta.json()
        
        if data['numFound'] == 0:
            return None
        
        autores = data['docs'][0].get('author_name')
        return autores[0] if autores else None
    except Exception as e:
        print(f"  ⚠️ Error título: {titulo[:30]} → {e}")
        return None

In [ ]:
encontrados    = 0
no_encontrados = 0

print(f"Procesando {len(libros_db)} libros...\n")

for i, (book_id, titulo) in enumerate(libros_db):
    
    if i % 100 == 0:
        print(f"  [{i}/{len(libros_db)}] procesando...")
    
    # Paso 1: buscar autor por título
    nombre_autor = buscar_autor_por_titulo(titulo)
    
    if not nombre_autor:
        datos = {
            "birth_year": None, "country": None,
            "external_api_id": None, "total_known_works": None,
            "api_source": "openlibrary+wikipedia", "api_status": "not_found"
        }
        nombre_autor = "Unknown"
        no_encontrados += 1
    else:
        # Paso 2: obtener datos completos con cache
        datos = obtener_datos_autor_con_cache(nombre_autor)
        encontrados += 1
    
    # Insertar autor y vincular al libro
    author_id = insertar_autor(nombre_autor, datos)
    vincular_libro_autor(book_id, author_id)
    
    time.sleep(0.3)

print(f"\n✅ Pipeline completo")
print(f"  Encontrados:    {encontrados}")
print(f"  No encontrados: {no_encontrados}")

In [46]:
cursor.execute("SELECT COUNT(*) FROM authors")
autores = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM book_author")
relaciones = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM authors WHERE api_status = 'found'")
encontrados = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM authors WHERE api_status = 'not_found'")
no_encontrados = cursor.fetchone()[0]

print("Estado final de la base de datos:")
print(f"  📚 Libros:            1000")
print(f"  📂 Categorías:        50")
print(f"  ✍️  Autores totales:   {autores}")
print(f"  ✅ Encontrados:       {encontrados}")
print(f"  ❌ No encontrados:    {no_encontrados}")
print(f"  🔗 Relaciones:        {relaciones}")

Estado final de la base de datos:
  📚 Libros:            1000
  📂 Categorías:        50
  ✍️  Autores totales:   552
  ✅ Encontrados:       542
  ❌ No encontrados:    10
  🔗 Relaciones:        1017


In [47]:
cursor.execute("""
    SELECT name, api_status 
    FROM authors 
    WHERE api_status = 'not_found'
""")

resultados = cursor.fetchall()

print("Autores no encontrados:")
for row in resultados:
    print(f"  - '{row[0]}'")

Autores no encontrados:
  - 'Unknown'
  - 'John H. Beauvais'
  - 'Melanie Dickerson'
  - 'Varsha Bajaj'
  - 'Megan Atwood'
  - 'Somme Sketcher'
  - 'Ernie Rosenberg'
  - 'Key Key Summaries'
  - 'Katie McGarry'
  - 'Bobbi Smith'


In [48]:
cursor.execute("""
    SELECT name 
    FROM authors 
    WHERE name LIKE '%â%' 
       OR name LIKE '%Â%'
       OR name LIKE '%ð%'
""")

resultados = cursor.fetchall()

print(f"Autores con posible problema de encoding: {len(resultados)}")
for row in resultados:
    print(f"  - '{row[0]}'")

Autores con posible problema de encoding: 0


Consulta 1 — Libros con más de 3 estrellas por menos de £10

In [49]:
cursor.execute("""
    SELECT 
        b.title,
        b.price,
        b.rating,
        c.name AS categoria
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.rating > 3
      AND b.price < 10.0
    ORDER BY b.rating DESC, b.price ASC
""")

resultados = cursor.fetchall()

print(f"Libros con rating > 3 y precio < £10: {len(resultados)}")
print()
for row in resultados:
    print(f"  [{row[2]}★] £{row[1]:.2f} — {row[0][:45]} ({row[3]})")

Libros con rating > 3 y precio < £10: 0



In [50]:
cursor.execute("""
    SELECT MIN(price), MAX(price), AVG(price)
    FROM books
""")
row = cursor.fetchone()
print(f"Precio mínimo: £{row[0]:.2f}")
print(f"Precio máximo: £{row[1]:.2f}")
print(f"Precio promedio: £{row[2]:.2f}")

Precio mínimo: £10.00
Precio máximo: £59.99
Precio promedio: £35.07


In [51]:
cursor.execute("""
    SELECT 
        b.title,
        b.price,
        b.rating,
        c.name AS categoria
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.rating > 3
      AND b.price < 20.0
    ORDER BY b.rating DESC, b.price ASC
""")

resultados = cursor.fetchall()

print(f"Libros con rating > 3 y precio < £20: {len(resultados)}")
print()
for row in resultados[:10]:
    print(f"  [{row[2]}★] £{row[1]:.2f} — {row[0][:45]} ({row[3]})")

Libros con rating > 3 y precio < £20: 75

  [5★] £10.00 — An Abundance of Katherines (Young Adult)
  [5★] £10.23 — Greek Mythic History (Default)
  [5★] £11.05 — The Power Greens Cookbook: 140 Delicious Supe (Food and Drink)
  [5★] £11.21 — Dear Mr. Knightley (Fiction)
  [5★] £11.33 — The Darkest Corners (Young Adult)
  [5★] £11.38 — Naturally Lean: 125 Nourishing Gluten-Free, P (Food and Drink)
  [5★] £11.64 — Fruits Basket, Vol. 2 (Fruits Basket #2) (Sequential Art)
  [5★] £11.83 — Old School (Diary of a Wimpy Kid #10) (Humor)
  [5★] £11.89 — Superman Vol. 1: Before Truth (Superman by Ge (Sequential Art)
  [5★] £12.16 — Every Heart a Doorway (Every Heart A Doorway  (Fantasy)


Hacemos commit y seguimos con la consulta 2:
Consulta 2 — Categoría con mayor precio promedio

In [52]:
cursor.execute("""
    SELECT 
        c.name AS categoria,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.price), 2) AS precio_promedio
    FROM categories c
    JOIN books b ON c.id = b.category_id
    GROUP BY c.id, c.name
    ORDER BY precio_promedio DESC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("Top 10 categorías por precio promedio:")
print()
for row in resultados:
    print(f"  £{row[2]:.2f} — {row[0]} ({row[1]} libros)")

Top 10 categorías por precio promedio:

  £58.33 — Suspense (1 libros)
  £54.81 — Novels (1 libros)
  £53.61 — Politics (3 libros)
  £51.45 — Health (4 libros)
  £46.38 — New Adult (6 libros)
  £42.50 — Christian (3 libros)
  £41.17 — Sports and Games (5 libros)
  £40.62 — Self Help (5 libros)
  £39.79 — Travel (11 libros)
  £39.59 — Fantasy (48 libros)


Consulta 3 — Autor con peor promedio de rating (mínimo 5 libros)


In [53]:
cursor.execute("""
    SELECT 
        a.name AS autor,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.rating), 2) AS promedio_rating
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    JOIN books b ON ba.book_id = b.id
    GROUP BY a.id, a.name
    HAVING COUNT(b.id) >= 5
    ORDER BY promedio_rating ASC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("Autores con peor promedio de rating (mínimo 5 libros):")
print()
for row in resultados:
    print(f"  {row[2]:.2f}★ — {row[0]} ({row[1]} libros)")

Autores con peor promedio de rating (mínimo 5 libros):

  2.00★ — Sophie Kinsella (7 libros)
  2.22★ — Worth Books (9 libros)
  2.33★ — J. K. Rowling (6 libros)
  2.67★ — Cassandra Clare (6 libros)
  2.75★ — Stephen King (12 libros)
  3.00★ — Dan Brown (5 libros)
  3.03★ — Unknown (355 libros)


Consulta 4 — Top 5 autores con más libros


In [54]:
cursor.execute("""
    SELECT 
        a.name AS autor,
        a.country AS pais,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.rating), 2) AS rating_promedio
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    JOIN books b ON ba.book_id = b.id
    GROUP BY a.id, a.name
    ORDER BY total_libros DESC
    LIMIT 5
""")

resultados = cursor.fetchall()

print("Top 5 autores con más libros:")
print()
for row in resultados:
    pais = row[1] if row[1] else "N/D"
    print(f"  {row[2]} libros — {row[0]} ({pais}) — {row[3]:.2f}★")

Top 5 autores con más libros:

  355 libros — Unknown (N/D) — 3.03★
  12 libros — Stephen King (American) — 2.75★
  9 libros — Worth Books (N/D) — 2.22★
  7 libros — Sophie Kinsella (English) — 2.00★
  6 libros — J. K. Rowling (British) — 2.33★


In [55]:
cursor.execute("""
    SELECT 
        a.name AS autor,
        a.country AS pais,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.rating), 2) AS rating_promedio
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    JOIN books b ON ba.book_id = b.id
    WHERE a.name != 'Unknown'
    GROUP BY a.id, a.name
    ORDER BY total_libros DESC
    LIMIT 5
""")

resultados = cursor.fetchall()

print("Top 5 autores con más libros (excluyendo Unknown):")
print()
for row in resultados:
    pais = row[1] if row[1] else "N/D"
    print(f"  {row[2]} libros — {row[0]} ({pais}) — {row[3]:.2f}★")

Top 5 autores con más libros (excluyendo Unknown):

  12 libros — Stephen King (American) — 2.75★
  9 libros — Worth Books (N/D) — 2.22★
  7 libros — Sophie Kinsella (English) — 2.00★
  6 libros — J. K. Rowling (British) — 2.33★
  6 libros — Cassandra Clare (American) — 2.67★


Consulta 5 ⭐ — La obligatoria: país con más libros de rating > 3Esta es la más importante del challenge. Requiere 3 JOINs:

In [56]:
cursor.execute("""
    SELECT 
        a.country AS pais,
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.rating), 2) AS rating_promedio
    FROM books b
    JOIN book_author ba ON b.id = ba.book_id
    JOIN authors a ON ba.author_id = a.id
    WHERE b.rating > 3
      AND a.country IS NOT NULL
      AND a.name != 'Unknown'
    GROUP BY a.country
    ORDER BY total_libros DESC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("País con más libros de rating > 3:")
print()
for row in resultados:
    print(f"  {row[1]} libros — {row[0]} — {row[2]:.2f}★ promedio")

País con más libros de rating > 3:

  108 libros — American — 4.57★ promedio
  15 libros — English — 4.40★ promedio
  13 libros — British — 4.46★ promedio
  9 libros — Canadian — 4.33★ promedio
  6 libros — Topics — 4.67★ promedio
  3 libros — Italian — 4.33★ promedio
  3 libros — Australian — 4.67★ promedio
  2 libros — French — 5.00★ promedio
  1 libros — Spanish — 5.00★ promedio
  1 libros — Singaporean — 5.00★ promedio


Indexación y Performance
Primero medimos el tiempo de una consulta SIN índice:

In [57]:
import time

# Consulta deliberadamente lenta - busca por rating y precio sin índice
consulta = """
    SELECT b.title, b.price, b.rating, c.name
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.rating > 3
      AND b.price < 40.0
    ORDER BY b.price ASC
"""

# Medimos 3 veces para tener un promedio más estable
tiempos = []
for i in range(3):
    inicio = time.perf_counter()
    cursor.execute(consulta)
    cursor.fetchall()
    fin = time.perf_counter()
    tiempos.append((fin - inicio) * 1000)

promedio_sin_indice = sum(tiempos) / len(tiempos)
print(f"Tiempo SIN índice:")
print(f"  Ejecución 1: {tiempos[0]:.3f} ms")
print(f"  Ejecución 2: {tiempos[1]:.3f} ms")
print(f"  Ejecución 3: {tiempos[2]:.3f} ms")
print(f"  Promedio:    {promedio_sin_indice:.3f} ms")

Tiempo SIN índice:
  Ejecución 1: 1.963 ms
  Ejecución 2: 1.421 ms
  Ejecución 3: 1.025 ms
  Promedio:    1.470 ms


Ahora creamos el índice y medimos de nuevo:

In [58]:
# Crear índice en las columnas que usa el WHERE
cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_rating ON books(rating)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_price ON books(price)")
conn.commit()

print("✅ Índices creados:")
print("   idx_books_rating → books(rating)")
print("   idx_books_price  → books(price)")

# Medimos de nuevo con el índice
tiempos = []
for i in range(3):
    inicio = time.perf_counter()
    cursor.execute(consulta)
    cursor.fetchall()
    fin = time.perf_counter()
    tiempos.append((fin - inicio) * 1000)

promedio_con_indice = sum(tiempos) / len(tiempos)
print(f"\nTiempo CON índice:")
print(f"  Ejecución 1: {tiempos[0]:.3f} ms")
print(f"  Ejecución 2: {tiempos[1]:.3f} ms")
print(f"  Ejecución 3: {tiempos[2]:.3f} ms")
print(f"  Promedio:    {promedio_con_indice:.3f} ms")

# Comparación final
mejora = ((promedio_sin_indice - promedio_con_indice) / promedio_sin_indice) * 100
print(f"\n📊 Resultado:")
print(f"  Sin índice:  {promedio_sin_indice:.3f} ms")
print(f"  Con índice:  {promedio_con_indice:.3f} ms")
print(f"  Mejora:      {mejora:.1f}%")

✅ Índices creados:
   idx_books_rating → books(rating)
   idx_books_price  → books(price)

Tiempo CON índice:
  Ejecución 1: 0.911 ms
  Ejecución 2: 0.656 ms
  Ejecución 3: 0.642 ms
  Promedio:    0.737 ms

📊 Resultado:
  Sin índice:  1.470 ms
  Con índice:  0.737 ms
  Mejora:      49.9%


¿Cómo explicás esto en la presentación?
"Sin índice, SQLite hace un Full Table Scan — lee los 1000 libros uno por uno para encontrar los que cumplen el WHERE. Con el índice, usa una estructura B-Tree que va directo a los registros relevantes sin leer todos. Con 1000 registros la mejora es del 50%. En una tabla con millones de filas la diferencia sería de segundos vs milisegundos."

https://dbdiagram.io/d/69d44e850f7c9ef2c0911c50
https://app.diagrams.net/?src=about

Arrancamos con la subconsulta.
Una subconsulta es un SELECT dentro de otro SELECT. Se usa cuando necesitás filtrar por un resultado que primero hay que calcular.
Subconsulta — Libros con precio mayor al promedio de su categoría

In [59]:
cursor.execute("""
    SELECT 
        b.title,
        b.price,
        c.name AS categoria,
        -- Subconsulta: calcula el promedio de precio de la categoría del libro
        (SELECT ROUND(AVG(b2.price), 2) 
         FROM books b2 
         WHERE b2.category_id = b.category_id) AS promedio_categoria
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.price > (
        -- Subconsulta en el WHERE: filtra libros más caros que el promedio de su categoría
        SELECT AVG(b3.price)
        FROM books b3
        WHERE b3.category_id = b.category_id
    )
    ORDER BY b.price DESC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("Libros con precio mayor al promedio de su categoría:")
print()
for row in resultados:
    print(f"  £{row[1]:.2f} (promedio: £{row[3]}) — {row[0][:40]} ({row[2]})")

Libros con precio mayor al promedio de su categoría:

  £59.99 (promedio: £33.93) — The Perfect Play (Play by Play #1) (Romance)
  £59.98 (promedio: £36.07) — Last One Home (New Beginnings #1) (Fiction)
  £59.95 (promedio: £34.22) — Civilization and Its Discontents (Psychology)
  £59.92 (promedio: £31.41) — The Barefoot Contessa Cookbook (Food and Drink)
  £59.90 (promedio: £34.26) — The Diary of a Young Girl (Nonfiction)
  £59.71 (promedio: £31.43) — The Bone Hunters (Lexy Vaughan & Steven  (Thriller)
  £59.64 (promedio: £37.29) — Thomas Jefferson and the Tripoli Pirates (History)
  £59.48 (promedio: £31.72) — Boar Island (Anna Pigeon #19) (Mystery)
  £59.45 (promedio: £34.26) — The Man Who Mistook His Wife for a Hat a (Nonfiction)
  £59.45 (promedio: £36.07) — The Improbability of Love (Fiction)


ora la función de ventana.
Una función de ventana opera sobre un conjunto de filas relacionadas sin colapsarlas como GROUP BY. La diferencia clave es que mantiene todas las filas y agrega información calculada sobre el grupo.
Función de ventana — Ranking de libros por precio dentro de cada categoría

In [61]:
cursor.execute("""
    SELECT title, price, categoria, ranking
    FROM (
        SELECT 
            b.title,
            b.price,
            c.name AS categoria,
            RANK() OVER (PARTITION BY b.category_id ORDER BY b.price DESC) AS ranking
        FROM books b
        JOIN categories c ON b.category_id = c.id
    )
    WHERE ranking = 1
    ORDER BY price DESC
    LIMIT 10
""")

resultados = cursor.fetchall()

print("Libro más caro por categoría:")
print()
for row in resultados:
    print(f"  #{row[3]} £{row[1]:.2f} — {row[0][:40]} ({row[2]})")

Libro más caro por categoría:

  #1 £59.99 — The Perfect Play (Play by Play #1) (Romance)
  #1 £59.98 — Last One Home (New Beginnings #1) (Fiction)
  #1 £59.95 — Civilization and Its Discontents (Psychology)
  #1 £59.92 — The Barefoot Contessa Cookbook (Food and Drink)
  #1 £59.90 — The Diary of a Young Girl (Nonfiction)
  #1 £59.71 — The Bone Hunters (Lexy Vaughan & Steven  (Thriller)
  #1 £59.64 — Thomas Jefferson and the Tripoli Pirates (History)
  #1 £59.48 — Boar Island (Anna Pigeon #19) (Mystery)
  #1 £59.15 — The Gray Rhino: How to Recognize and Act (Add a comment)
  #1 £59.04 — Life Without a Recipe (Autobiography)


Ahora la prueba de que los datos provienen de la API:

In [62]:
cursor.execute("""
    SELECT 
        a.name,
        a.birth_year,
        a.country,
        a.external_api_id,
        a.total_known_works,
        a.api_source
    FROM authors a
    WHERE a.api_status = 'found'
      AND a.country IS NOT NULL
      AND a.external_api_id IS NOT NULL
    LIMIT 10
""")

resultados = cursor.fetchall()

print("Prueba de datos enriquecidos por la API:")
print("-" * 70)
for row in resultados:
    print(f"  Autor:      {row[0]}")
    print(f"  Nacimiento: {row[1]}")
    print(f"  País:       {row[2]}")
    print(f"  ID externo: {row[3]}")
    print(f"  Obras:      {row[4]}")
    print(f"  Fuente:     {row[5]}")
    print()

Prueba de datos enriquecidos por la API:
----------------------------------------------------------------------
  Autor:      Gillian Flynn
  Nacimiento: 1971
  País:       American
  ID externo: OL1433006A
  Obras:      41
  Fuente:     openlibrary+wikipedia

  Autor:      Frances Mayes
  Nacimiento: None
  País:       American
  ID externo: OL7412363A
  Obras:      33
  Fuente:     openlibrary+wikipedia

  Autor:      Paul Theroux
  Nacimiento: 1941
  País:       American
  ID externo: OL4416468A
  Obras:      241
  Fuente:     openlibrary+wikipedia

  Autor:      Patricia Schultz
  Nacimiento: None
  País:       American
  ID externo: OL2625971A
  Obras:      61
  Fuente:     openlibrary+wikipedia

  Autor:      Ruth Ware
  Nacimiento: 1977
  País:       British
  ID externo: OL7764121A
  Obras:      50
  Fuente:     openlibrary+wikipedia

  Autor:      Celeste Bradley
  Nacimiento: None
  País:       American
  ID externo: OL1422811A
  Obras:      31
  Fuente:     openlibrary+wikip

In [ ]:
cursor.execute("SELECT COUNT(*) FROM books")
print(cursor.fetchone()[0])